In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Khởi tạo Spark Session
spark = SparkSession.builder.appName("Green").getOrCreate()

print("Spark Session đã được khởi tạo thành công!")


Spark Session đã được khởi tạo thành công!


In [2]:
green = spark.read.parquet("hdfs://master:9000/data/parquet/green_tripdata_2024-*.parquet")

In [3]:
from pyspark.sql.functions import (
    col, to_date, hour, dayofweek, dayofmonth, month, count, when, lag,
    desc, sum as _sum, avg, unix_timestamp
)
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, IndexToString
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
green_filtered = green.filter(
    (col("lpep_pickup_datetime").isNotNull()) &
    (col("lpep_dropoff_datetime").isNotNull()) &
    (col("PULocationID").isNotNull()) &
    (col("DOLocationID").isNotNull())
)

# Lọc các giá trị PULocationID và DOLocationID hợp lệ
green_filtered = green_filtered.filter(
    (col("PULocationID") > 0) & (col("PULocationID") <= 263) &
    (col("DOLocationID") > 0) & (col("DOLocationID") <= 263)
)

# Lọc bỏ các chuyến đi có thời gian và quãng đường phi lý
green_filtered = green_filtered.withColumn(
    "calculated_trip_time_seconds",
    unix_timestamp(col("lpep_dropoff_datetime")) - unix_timestamp(col("lpep_pickup_datetime"))
)

green_filtered = green_filtered.withColumn(
    "calculated_average_speed_mph",
    (col("trip_distance") / col("calculated_trip_time_seconds")) * 3600
)

green_filtered = green_filtered.filter(
    (col("calculated_trip_time_seconds") >= 60) & (col("calculated_trip_time_seconds") <= 18000) &
    (col("trip_distance") >= 0.1) & (col("trip_distance") <= 50) &
    (col("calculated_average_speed_mph") >= 1) & (col("calculated_average_speed_mph") <= 60)
)

# Lọc các giá trị tài chính hợp lệ
green_filtered = green_filtered.filter(
    (col("passenger_count") > 0) &
    (col("extra") >= 0) & (col("tip_amount") >= 0) &
    (col("tolls_amount") >= 0) & (col("total_amount") > 0) &
    (col("congestion_surcharge") >= 0) & (col("congestion_surcharge") < 20)
)

# Gán lại vào biến green để sử dụng cho các bước sau
green = green_filtered
print(f"Tổng số dòng của Green Taxi sau khi lọc: {green.count()}")


Tổng số dòng của Green Taxi sau khi lọc: 579676


In [5]:
print("\n--- BƯỚC 1: Đang tổng hợp dữ liệu từ 'green' thành chuỗi thời gian theo giờ... ---")
# Sử dụng cột thời gian của Green Taxi: lpep_pickup_datetime
hourly_demand = green.groupBy(
    to_date(col("lpep_pickup_datetime")).alias("date"), 
    hour(col("lpep_pickup_datetime")).alias("hour"),
    col("PULocationID")
).agg(count("*").alias("trip_count"))

print("\n--- BƯỚC 2: Đang tạo đặc trưng và phân cụm các địa điểm... ---")
# a. Tạo các đặc trưng mô tả "tính cách" của mỗi địa điểm
total_trips_per_loc = hourly_demand.groupBy("PULocationID").agg(_sum("trip_count").alias("total_trips"))
avg_trips_per_hour_loc = hourly_demand.groupBy("PULocationID").agg(avg("trip_count").alias("avg_trips_per_hour"))
weekend_ratio_loc = hourly_demand.withColumn("day_of_week", dayofweek(col("date"))) \
                                 .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), 1).otherwise(0)) \
                                 .groupBy("PULocationID") \
                                 .agg((_sum("is_weekend") / count("*")).alias("weekend_ratio"))
location_features_df = total_trips_per_loc.join(avg_trips_per_hour_loc, "PULocationID").join(weekend_ratio_loc, "PULocationID")

# b. Chuẩn bị và chạy mô hình K-Means để phân cụm
feature_cols_for_clustering = ["total_trips", "avg_trips_per_hour", "weekend_ratio"]
assembler_for_clustering = VectorAssembler(inputCols=feature_cols_for_clustering, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)
kmeans = KMeans(featuresCol="features", k=5, seed=42)
clustering_pipeline = Pipeline(stages=[assembler_for_clustering, scaler, kmeans])
clustering_model = clustering_pipeline.fit(location_features_df)
locations_with_clusters = clustering_model.transform(location_features_df).select(col("PULocationID"), col("prediction").alias("cluster_id"))
print("Đã phân các địa điểm thành 5 cụm.")
locations_with_clusters.groupBy("cluster_id").count().show()


--- BƯỚC 1: Đang tổng hợp dữ liệu từ 'green' thành chuỗi thời gian theo giờ... ---

--- BƯỚC 2: Đang tạo đặc trưng và phân cụm các địa điểm... ---
Đã phân các địa điểm thành 5 cụm.
+----------+-----+
|cluster_id|count|
+----------+-----+
|         1|    2|
|         3|   18|
|         4|  117|
|         2|    9|
|         0|   95|
+----------+-----+



In [6]:
print("\n--- BƯỚC 3: Đang chuẩn bị dữ liệu hoàn chỉnh để huấn luyện... ---")
# a. Thêm các đặc trưng thời gian
model_data = hourly_demand.withColumn("day_of_week", dayofweek(col("date"))) \
                          .withColumn("day_of_month", dayofmonth(col("date"))) \
                          .withColumn("month", month(col("date")))
# b. Thêm đặc trưng Kỳ nghỉ
holiday_list = [("2024-01-01", "Holiday"), ("2024-01-15", "Holiday"), ("2024-05-27", "Holiday"), ("2024-07-04", "Holiday"), ("2024-09-02", "Holiday"), ("2024-11-28", "Holiday"), ("2024-12-25", "Holiday")]
holiday_periods_list = []
for holiday_date_str, type in holiday_list:
    holiday_date = pd.to_datetime(holiday_date_str)
    holiday_periods_list.append((str(holiday_date.date()), type))
    holiday_periods_list.append((str((holiday_date - pd.DateOffset(days=1)).date()), "Pre-Holiday"))
    holiday_periods_list.append((str((holiday_date + pd.DateOffset(days=1)).date()), "Post-Holiday"))
holiday_periods_df = spark.createDataFrame(holiday_periods_list, ["date_str", "day_type"]).withColumn("date", to_date(col("date_str"))).drop("date_str")
model_data_with_holidays = model_data.join(holiday_periods_df, ["date"], "left").na.fill({"day_type": "Normal_Day"})
# c. Thêm đặc trưng Trễ
window_spec = Window.partitionBy("PULocationID").orderBy("date", "hour")
data_with_features = model_data_with_holidays.withColumn("lag_24hr", lag("trip_count", 24).over(window_spec)) \
                                             .withColumn("lag_168hr", lag("trip_count", 24 * 7).over(window_spec)) \
                                             .na.fill(0, subset=["lag_24hr", "lag_168hr"])
# d. Thêm ID cụm vào dữ liệu
data_with_clusters = data_with_features.join(locations_with_clusters, "PULocationID", "left")
# e. Tạo nhãn phân loại và trọng số
quantiles = data_with_clusters.approxQuantile("trip_count", [0.25, 0.5, 0.75, 0.95], 0.01)
data_with_levels = data_with_clusters.withColumn("demand_level",
    when(col("trip_count") == 0, "None").when(col("trip_count") <= quantiles[0], "Very_Low").when(col("trip_count") <= quantiles[1], "Low")
    .when(col("trip_count") <= quantiles[2], "Medium").when(col("trip_count") <= quantiles[3], "High").otherwise("Very_High"))
class_counts = data_with_levels.groupBy("demand_level").count()
total_samples = data_with_levels.count()
num_classes = class_counts.count()
calculate_weight = class_counts.withColumn("weight", total_samples / (num_classes * col("count")))
final_data_for_training = data_with_levels.join(calculate_weight.select("demand_level", "weight"), ["demand_level"], "left")

# ==============================================================================


--- BƯỚC 3: Đang chuẩn bị dữ liệu hoàn chỉnh để huấn luyện... ---


In [7]:
print("\n--- BƯỚC 4: Bắt đầu huấn luyện mô hình chuyên biệt cho từng cụm... ---")
# a. Định nghĩa pipeline huấn luyện (sẽ được tái sử dụng)
label_indexer = StringIndexer(inputCol="demand_level", outputCol="label", handleInvalid="keep")
day_type_indexer = StringIndexer(inputCol="day_type", outputCol="day_type_index", handleInvalid="keep")
day_type_encoder = OneHotEncoder(inputCol="day_type_index", outputCol="day_type_vec")
feature_columns = ["PULocationID", "hour", "day_of_week", "day_of_month", "month", "lag_24hr","lag_168hr", "day_type_vec"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="skip")
rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42)
training_pipeline = Pipeline(stages=[label_indexer, day_type_indexer, day_type_encoder, assembler, rf_classifier])

# b. Lặp qua từng cụm để huấn luyện và đánh giá
unique_clusters = [row.cluster_id for row in locations_with_clusters.select("cluster_id").distinct().collect()]
trained_models = {}

for cluster_id in sorted(unique_clusters):
    print(f"\n===== Đang xử lý Cụm số: {cluster_id} =====")
    
    cluster_data = final_data_for_training.filter(col("cluster_id") == cluster_id)
    train_cluster, test_cluster = cluster_data.randomSplit([0.8, 0.2], seed=42)
    
    if train_cluster.count() < 20: 
        print(f"!!! CẢNH BÁO: Cụm {cluster_id} có quá ít dữ liệu để huấn luyện. Bỏ qua cụm này.")
        continue

    print(f"Huấn luyện mô hình cho cụm {cluster_id}...")
    model_for_cluster = training_pipeline.fit(train_cluster)
    trained_models[cluster_id] = model_for_cluster
    
    if test_cluster.count() > 0:
        predictions = model_for_cluster.transform(test_cluster)
        evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
        f1 = evaluator_f1.evaluate(predictions)
        print(f"==> KẾT QUẢ CHO CỤM {cluster_id}: F1-Score = {f1:.4f}")
    else:
        print(f"Cụm {cluster_id} không có dữ liệu trong tập test để đánh giá.")


--- BƯỚC 4: Bắt đầu huấn luyện mô hình chuyên biệt cho từng cụm... ---

===== Đang xử lý Cụm số: 0 =====
Huấn luyện mô hình cho cụm 0...
==> KẾT QUẢ CHO CỤM 0: F1-Score = 0.5029

===== Đang xử lý Cụm số: 1 =====
Huấn luyện mô hình cho cụm 1...
==> KẾT QUẢ CHO CỤM 1: F1-Score = 0.5873

===== Đang xử lý Cụm số: 2 =====
Huấn luyện mô hình cho cụm 2...
==> KẾT QUẢ CHO CỤM 2: F1-Score = 0.3505

===== Đang xử lý Cụm số: 3 =====
Huấn luyện mô hình cho cụm 3...
==> KẾT QUẢ CHO CỤM 3: F1-Score = 0.5990

===== Đang xử lý Cụm số: 4 =====
Huấn luyện mô hình cho cụm 4...
==> KẾT QUẢ CHO CỤM 4: F1-Score = 0.7111


In [8]:
from pyspark.sql.functions import col, max, min, avg


cluster_to_investigate = 2
# ----------------------------------------------------

print(f"--- Đang tính toán thống kê (max, min, avg) cho Cụm {cluster_to_investigate} ---")

# 1. Lọc ra dữ liệu chỉ thuộc về Cụm 2
#    Giả định DataFrame `data_with_clusters` đã tồn tại
cluster_data = data_with_clusters.filter(col("cluster_id") == cluster_to_investigate)

# 2. Sử dụng .agg() để tính toán các giá trị thống kê trên cột 'trip_count'
cluster_stats = cluster_data.agg(
    max("trip_count").alias("max_trip_count"),
    min("trip_count").alias("min_trip_count"),
    avg("trip_count").alias("avg_trip_count")
)

# 3. Hiển thị kết quả
print(f"\nBảng thống kê số lượng chuyến đi cho Cụm {cluster_to_investigate}:")
cluster_stats.show()

--- Đang tính toán thống kê (max, min, avg) cho Cụm 2 ---

Bảng thống kê số lượng chuyến đi cho Cụm 2:
+--------------+--------------+----------------+
|max_trip_count|min_trip_count|  avg_trip_count|
+--------------+--------------+----------------+
|            39|             1|4.14742583257582|
+--------------+--------------+----------------+



In [9]:
from pyspark.sql.functions import col, max, min, avg
from pyspark.ml.feature import IndexToString

# --- Thay đổi ID của cụm bạn muốn chẩn đoán ở đây ---
cluster_to_investigate = 0
# ----------------------------------------------------

print(f"============================================================")
print(f"Tính toán max, min, avg: {cluster_to_investigate}")
print(f"============================================================")

# Giả định các DataFrame `final_data_for_training` và `training_pipeline` đã tồn tại

# 1. Lọc ra dữ liệu chỉ thuộc về Cụm 0
cluster_data = final_data_for_training.filter(col("cluster_id") == cluster_to_investigate)

# 2. Xem lại thống kê nhu cầu của Cụm 0
print(f"\n--- Thống kê nhu cầu cho Cụm {cluster_to_investigate} ---")
cluster_stats = cluster_data.agg(
    max("trip_count").alias("max_trip_count"),
    min("trip_count").alias("min_trip_count"),
    avg("trip_count").alias("avg_trip_count")
)
cluster_stats.show()


Tính toán max, min, avg: 0

--- Thống kê nhu cầu cho Cụm 0 ---
+--------------+--------------+------------------+
|max_trip_count|min_trip_count|    avg_trip_count|
+--------------+--------------+------------------+
|            19|             1|1.6895045653822065|
+--------------+--------------+------------------+



In [10]:
from pyspark.sql.functions import col, max, min, avg
from pyspark.ml.feature import IndexToString

# --- Thay đổi ID của cụm bạn muốn chẩn đoán ở đây ---
cluster_to_investigate = 1
# ----------------------------------------------------

print(f"============================================================")
print(f"Tính toán max, min, avg: {cluster_to_investigate}")
print(f"============================================================")

# Giả định các DataFrame `final_data_for_training` và `training_pipeline` đã tồn tại

# 1. Lọc ra dữ liệu chỉ thuộc về Cụm 0
cluster_data = final_data_for_training.filter(col("cluster_id") == cluster_to_investigate)

# 2. Xem lại thống kê nhu cầu của Cụm 0
print(f"\n--- Thống kê nhu cầu cho Cụm {cluster_to_investigate} ---")
cluster_stats = cluster_data.agg(
    max("trip_count").alias("max_trip_count"),
    min("trip_count").alias("min_trip_count"),
    avg("trip_count").alias("avg_trip_count")
)
cluster_stats.show()





Tính toán max, min, avg: 1

--- Thống kê nhu cầu cho Cụm 1 ---
+--------------+--------------+------------------+
|max_trip_count|min_trip_count|    avg_trip_count|
+--------------+--------------+------------------+
|            68|             1|15.207299270072992|
+--------------+--------------+------------------+



In [11]:
from pyspark.sql.functions import col, max, min, avg
from pyspark.ml.feature import IndexToString

# --- Thay đổi ID của cụm bạn muốn chẩn đoán ở đây ---
cluster_to_investigate = 3
# ----------------------------------------------------

print(f"============================================================")
print(f"Tính toán max, min, avg: {cluster_to_investigate}")
print(f"============================================================")

# Giả định các DataFrame `final_data_for_training` và `training_pipeline` đã tồn tại

# 1. Lọc ra dữ liệu chỉ thuộc về Cụm 0
cluster_data = final_data_for_training.filter(col("cluster_id") == cluster_to_investigate)

# 2. Xem lại thống kê nhu cầu của Cụm 0
print(f"\n--- Thống kê nhu cầu cho Cụm {cluster_to_investigate} ---")
cluster_stats = cluster_data.agg(
    max("trip_count").alias("max_trip_count"),
    min("trip_count").alias("min_trip_count"),
    avg("trip_count").alias("avg_trip_count")
)
cluster_stats.show()


Tính toán max, min, avg: 3

--- Thống kê nhu cầu cho Cụm 3 ---
+--------------+--------------+------------------+
|max_trip_count|min_trip_count|    avg_trip_count|
+--------------+--------------+------------------+
|            25|             1|2.1419686896847523|
+--------------+--------------+------------------+



In [12]:
from pyspark.sql.functions import col, max, min, avg
from pyspark.ml.feature import IndexToString

# --- Thay đổi ID của cụm bạn muốn chẩn đoán ở đây ---
cluster_to_investigate = 4
# ----------------------------------------------------

print(f"============================================================")
print(f"Tính toán max, min, avg: {cluster_to_investigate}")
print(f"============================================================")

# Giả định các DataFrame `final_data_for_training` và `training_pipeline` đã tồn tại

# 1. Lọc ra dữ liệu chỉ thuộc về Cụm 0
cluster_data = final_data_for_training.filter(col("cluster_id") == cluster_to_investigate)

# 2. Xem lại thống kê nhu cầu của Cụm 0
print(f"\n--- Thống kê nhu cầu cho Cụm {cluster_to_investigate} ---")
cluster_stats = cluster_data.agg(
    max("trip_count").alias("max_trip_count"),
    min("trip_count").alias("min_trip_count"),
    avg("trip_count").alias("avg_trip_count")
)
cluster_stats.show()



from pyspark.sql.functions import col, collect_list, sort_array

# Giả định DataFrame `locations_with_clusters` đã tồn tại từ bước phân cụm

# Nhóm theo ID cụm và thu thập tất cả các PULocationID vào một danh sách
locations_per_cluster = locations_with_clusters.groupBy("cluster_id") \
                                               .agg(sort_array(collect_list("PULocationID")).alias("location_ids")) \
                                               .orderBy("cluster_id")

# Hiển thị kết quả
locations_per_cluster.show(truncate=False)

# Để lấy ra danh sách Python cho một cụm cụ thể (ví dụ: Cụm 0)
# cluster_0_list = locations_per_cluster.filter(col("cluster_id") == 0).first()["location_ids"]
# print(cluster_0_list)

Tính toán max, min, avg: 4

--- Thống kê nhu cầu cho Cụm 4 ---
+--------------+--------------+------------------+
|max_trip_count|min_trip_count|    avg_trip_count|
+--------------+--------------+------------------+
|            11|             1|1.3138997636934364|
+--------------+--------------+------------------+

+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|cluster_id|location_ids                                                                                                   

In [13]:
# --- BƯỚC 1: Tự động tính toán và lưu giá trị max_trip_count cho mỗi cụm ---
# Tính toán các giá trị max cho mỗi cụm từ dữ liệu đã có
cluster_stats_df = final_data_for_training.groupBy("cluster_id").agg(
    max("trip_count").alias("max_trip_count")
)

# Chuyển kết quả thành một dictionary để tra cứu nhanh: {cluster_id: max_value}
cluster_max_values = {row['cluster_id']: row['max_trip_count'] for row in cluster_stats_df.collect()}

print("Đã lưu giá trị max trip count cho mỗi cụm:")
print(cluster_max_values)


# --- BƯỚC 2: Cập nhật lại hàm get_recommendation để sử dụng dictionary ---
def get_recommendation_fixed(location_id, date_str, hour_of_day):
    """
    Hàm gợi ý đã được sửa lỗi NameError bằng cách tra cứu giá trị max_trip_count
    từ dictionary đã được tạo sẵn.
    """
    print(f"--- Đang phân tích cho LocationID: {location_id}, Thời gian: {hour_of_day}h ngày {date_str} ---")

    # 1. Xác định Cụm
    cluster_info = locations_with_clusters.filter(col("PULocationID") == location_id).first()
    if not cluster_info:
        return f"Lỗi: Không tìm thấy LocationID {location_id}."
    cluster_id = cluster_info["cluster_id"]
    print(f"LocationID {location_id} thuộc Cụm số: {cluster_id}")
    
    # Lấy giá trị max_trip_count cho cụm này
    max_trip_count_for_cluster = cluster_max_values.get(cluster_id, 0) # Mặc định là 0 nếu không tìm thấy

    # 2. Áp dụng Logic Quy tắc (nếu có)
    if cluster_id in [0, 2]: # Giả sử bạn muốn giữ quy tắc cho cụm 0 và 2
         # Lấy khoảng giá trị từ phân tích trước đó
        min_val = 1
        max_val = max_trip_count_for_cluster
        return f"Gợi ý cho Cụm {cluster_id} (Dựa trên quy tắc): Nhu cầu ước tính thấp-trung bình, cần khoảng {min_val} đến {max_val} chuyến đi."

    # 3. Áp dụng Logic Mô hình Học máy
    print(f"Áp dụng mô hình học máy cho Cụm {cluster_id}...")
    
    # Chuẩn bị dữ liệu đầu vào (tương tự code cũ)
    input_data = spark.createDataFrame([(location_id, date_str, hour_of_day)], ["PULocationID", "date_str", "hour"])
    input_data = input_data.withColumn("date", to_date(col("date_str"))) \
                           .withColumn("day_of_week", dayofweek(col("date"))) \
                           .withColumn("day_of_month", dayofmonth(col("date"))) \
                           .withColumn("month", month(col("date")))
    input_data = input_data.join(holiday_periods_df, ["date"], "left").na.fill({"day_type": "Normal_Day"})
    
    previous_day_date = pd.to_datetime(date_str) - pd.DateOffset(days=1)
    lag_24hr_row = hourly_demand.filter((col("PULocationID") == location_id) & (to_date(col("date")) == lit(previous_day_date.strftime('%Y-%m-%d'))) & (col("hour") == hour_of_day)).first()
    input_data = input_data.withColumn("lag_24hr", lit(lag_24hr_row["trip_count"] if lag_24hr_row else 0))

    previous_week_date = pd.to_datetime(date_str) - pd.DateOffset(days=7)
    lag_168hr_row = hourly_demand.filter((col("PULocationID") == location_id) & (to_date(col("date")) == lit(previous_week_date.strftime('%Y-%m-%d'))) & (col("hour") == hour_of_day)).first()
    input_data = input_data.withColumn("lag_168hr", lit(lag_168hr_row["trip_count"] if lag_168hr_row else 0))

    # Dự đoán và diễn giải kết quả
    if cluster_id in trained_models:
        model_to_use = trained_models[cluster_id]
        labels = model_to_use.stages[0].labels
        prediction_converter = IndexToString(inputCol="prediction", outputCol="predicted_level", labels=labels)
        prediction_result = model_to_use.transform(input_data)
        predicted_level_str = prediction_converter.transform(prediction_result).first()["predicted_level"]
        
        # Diễn giải kết quả
        if predicted_level_str == "Very_Low":
            range_str = f"khoảng 1 đến {int(quantiles[0])} xe"
        elif predicted_level_str == "Low":
            range_str = f"khoảng {int(quantiles[0]) + 1} đến {int(quantiles[1])} xe"
        elif predicted_level_str == "Medium":
            range_str = f"khoảng {int(quantiles[1]) + 1} đến {int(quantiles[2])} xe"
        elif predicted_level_str == "High":
            range_str = f"khoảng {int(quantiles[2]) + 1} đến {int(quantiles[3])} xe"
        else: # Very_High
            # SỬ DỤNG BIẾN ĐÃ ĐƯỢC GÁN Ở ĐẦU HÀM
            range_str = f"khoảng {int(quantiles[3]) + 1} đến {max_trip_count_for_cluster} xe"
            
        return f"Dự đoán cho Cụm {cluster_id}: Mức độ nhu cầu là '{predicted_level_str}'. Gợi ý số lượng xe: {range_str}."
    else:
        return f"Lỗi: Không tìm thấy mô hình đã huấn luyện cho Cụm {cluster_id}."

Đã lưu giá trị max trip count cho mỗi cụm:
{1: 68, 3: 25, 4: 11, 2: 39, 0: 19}


In [15]:
from pyspark.sql.functions import lit, broadcast, to_date, dayofweek, dayofmonth, month, col
from functools import reduce
from pyspark.ml.feature import IndexToString

print("\n--- BẮT ĐẦU DỰ ĐOÁN CHO TOÀN BỘ GREEN TAXI VÀO NGÀY 1/1/2025 ---")

# --- Bước 1: Tạo lưới dữ liệu cho ngày dự đoán (tương tự Yellow) ---
hours_df = spark.range(0, 24).withColumnRenamed("id", "hour")
locations_df = spark.range(1, 264).withColumnRenamed("id", "PULocationID").withColumn("PULocationID", col("PULocationID").cast("integer"))
prediction_grid = locations_df.crossJoin(broadcast(hours_df))

# --- Bước 2: Thêm các đặc trưng thời gian, kỳ nghỉ và CỤM ---
target_date = "2025-01-01"
future_data = prediction_grid.withColumn("date", to_date(lit(target_date))) \
                             .withColumn("day_of_week", dayofweek(col("date"))) \
                             .withColumn("day_of_month", dayofmonth(col("date"))) \
                             .withColumn("month", month(col("date"))) \
                             .withColumn("day_type", lit("Holiday"))

# Thêm thông tin CỤM vào lưới dữ liệu
# Giả định `locations_with_clusters` đã tồn tại
future_data = future_data.join(locations_with_clusters, "PULocationID", "left")

# --- Bước 3: Thêm dữ liệu quá khứ (Lag Features) ---
# Giả định `hourly_demand` của Green Taxi đã tồn tại
lag_24hr_data = hourly_demand.filter(col("date") == "2024-12-31").select("PULocationID", "hour", col("trip_count").alias("lag_24hr"))
lag_168hr_data = hourly_demand.filter(col("date") == "2024-12-25").select("PULocationID", "hour", col("trip_count").alias("lag_168hr"))

future_data = future_data.join(lag_24hr_data, ["PULocationID", "hour"], "left") \
                         .join(lag_168hr_data, ["PULocationID", "hour"], "left") \
                         .na.fill(0, subset=["lag_24hr", "lag_168hr", "cluster_id"])

print("Đã tạo xong dữ liệu đầu vào cho ngày 1/1/2025.")

# --- Bước 4: Lặp qua từng cụm để áp dụng mô hình/quy tắc phù hợp ---
all_predictions_list = []
unique_clusters_future = [row.cluster_id for row in future_data.select("cluster_id").distinct().collect()]

for cid in sorted(unique_clusters_future):
    print(f"Đang xử lý dự đoán cho Cụm {cid}...")
    cluster_subset = future_data.filter(col("cluster_id") == cid)
    
    # Giả định `trained_models` và `cluster_max_values` đã tồn tại
    if cid in trained_models:
        model_to_use = trained_models[cid]
        predictions_for_cluster = model_to_use.transform(cluster_subset)
        
        labels = model_to_use.stages[0].labels
        converter = IndexToString(inputCol="prediction", outputCol="predicted_level", labels=labels)
        final_predictions_for_cluster = converter.transform(predictions_for_cluster)
        
        # Thêm các cột cần thiết để join sau này
        all_predictions_list.append(final_predictions_for_cluster.select("PULocationID", "hour", "predicted_level"))
    
    elif cid in [0, 2]:
        min_val = 1
        max_val = cluster_max_values.get(cid, 0)
        rule_based_prediction = cluster_subset.withColumn("predicted_level", lit(f"Rule: Low-Med ({min_val}-{max_val})"))
        all_predictions_list.append(rule_based_prediction.select("PULocationID", "hour", "predicted_level"))

# --- Bước 5: Ghép tất cả các kết quả dự đoán lại ---
if all_predictions_list:
    final_full_city_predictions = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), all_predictions_list)
    
    print("\n--- KẾT QUẢ DỰ ĐOÁN CHO TOÀN THÀNH PHỐ NGÀY 1/1/2025 ---")
    
    # Thống kê tổng quan
    print("\n>>> Tổng quan dự đoán trên toàn thành phố:")
    final_full_city_predictions.groupBy("predicted_level").count().orderBy(col("count").desc()).show(50)
else:
    print("Không có dự đoán nào được tạo ra.")

# ==============================================================================
# --- Bước 6: Chuẩn bị và xuất kết quả dự đoán ra file CSV (ĐÃ CHỈNH SỬA) ---

print("\n--- Đang chuẩn bị và xuất kết quả dự đoán của Green Taxi ra file CSV ---")

# Join kết quả dự đoán với lưới dữ liệu ban đầu để lấy lại cột `date` và `day_of_week`
final_output_with_dates = final_full_city_predictions.join(
    future_data.select("PULocationID", "hour", "date", "day_of_week"),
    ["PULocationID", "hour"],
    "inner"
)

# Chọn chính xác các cột bạn muốn và sắp xếp lại
output_df = final_output_with_dates.select(
    "PULocationID",
    "hour",
    "date",
    "day_of_week",
    "predicted_level"
).orderBy("PULocationID", "hour")

print("\n>>> Dữ liệu mẫu sẽ được xuất ra file CSV:")
output_df.show(10)

# Ghi DataFrame đã được định dạng ra file CSV
output_path = "hdfs://master:9000/output/green_predictions_2025-01-01_formatted"
output_df.coalesce(1) \
         .write \
         .option("header", "true") \
         .mode("overwrite") \
         .csv(output_path)

print(f"Đã xuất file CSV thành công vào thư mục: {output_path}")
# ==============================================================================


--- BẮT ĐẦU DỰ ĐOÁN CHO TOÀN BỘ GREEN TAXI VÀO NGÀY 1/1/2025 ---
Đã tạo xong dữ liệu đầu vào cho ngày 1/1/2025.
Đang xử lý dự đoán cho Cụm 0...
Đang xử lý dự đoán cho Cụm 1...
Đang xử lý dự đoán cho Cụm 2...
Đang xử lý dự đoán cho Cụm 3...
Đang xử lý dự đoán cho Cụm 4...

--- KẾT QUẢ DỰ ĐOÁN CHO TOÀN THÀNH PHỐ NGÀY 1/1/2025 ---

>>> Tổng quan dự đoán trên toàn thành phố:
+---------------+-----+
|predicted_level|count|
+---------------+-----+
|       Very_Low| 5998|
|            Low|  128|
|         Medium|   79|
|           High|   74|
|      Very_High|   33|
+---------------+-----+


--- Đang chuẩn bị và xuất kết quả dự đoán của Green Taxi ra file CSV ---

>>> Dữ liệu mẫu sẽ được xuất ra file CSV:
+------------+----+----------+-----------+---------------+
|PULocationID|hour|      date|day_of_week|predicted_level|
+------------+----+----------+-----------+---------------+
|           1|   0|2025-01-01|          4|       Very_Low|
|           1|   1|2025-01-01|          4|       Very_L